# **01: Python Data Audit and Cleaning**

**Module:** Databases and Analytics (CP6UA56O)  
**Case study:** NorthStar Urban Mobility and Logistics

This notebook covers the dataset import, audit, cleaning, feature engineering, and consolidated dataframe preparation stages. Raw CSV files are imported from the GitHub repository, cleaned with Python and Pandas, and saved as cleaned output files for the later SQL, R, Python analytics, and MongoDB sections.

---
## **Install and Import Libraries**

The main Python libraries required for data inspection and cleaning are imported. These libraries are already available in Google Colab, so no extra installation is usually required.

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import os

# Suppress unnecessary warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Display settings: show all columns in output tables
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print('Libraries loaded successfully.')

Libraries loaded successfully.


---
## **Load Raw Dataset from GitHub**

The raw NorthStar CSV files were uploaded to the GitHub repository first. In this notebook, the files are imported directly from GitHub into Google Colab using raw GitHub URLs.

This approach supports the coursework requirement because the dataset is stored in GitHub and then imported into Colab for analysis and cleaning.

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

GITHUB_BASE_URL = 'https://raw.githubusercontent.com/SamsonSiby5827/Northstar_Databases_Analytics_Coursework/main/Data/Raw'


print('GitHub raw data source:')
print(GITHUB_BASE_URL)


GitHub raw data source:
https://raw.githubusercontent.com/SamsonSiby5827/Northstar_Databases_Analytics_Coursework/main/Data/Raw


---
## **Load All 10 CSV Files**

Each CSV file is loaded into a separate Pandas DataFrame. After loading, the number of rows and columns in each file is checked. This gives a quick confirmation that the files have been imported correctly.

In [ ]:
files_to_load = {
    'app_events': 'app_events.csv',
    'complaints': 'complaints.csv',
    'customers': 'customers.csv',
    'data_dictionary': 'data_dictionary.csv',
    'deliveries': 'deliveries.csv',
    'drivers': 'drivers.csv',
    'hubs': 'hubs.csv',
    'incidents': 'incidents.csv',
    'orders': 'orders.csv',
    'vehicles': 'vehicles.csv'
}

data = {}

for name, filename in files_to_load.items():
    file_url = f'{GITHUB_BASE_URL}/{filename}'
    data[name] = pd.read_csv(file_url)
    print(f'{name}: {data[name].shape[0]} rows, {data[name].shape[1]} columns loaded from GitHub')

# Create individual dataframe names so the rest of the notebook can use clear variable names.
app_events = data['app_events']
complaints = data['complaints']
customers = data['customers']
data_dict = data['data_dictionary']
deliveries = data['deliveries']
drivers = data['drivers']
hubs = data['hubs']
incidents = data['incidents']
orders = data['orders']
vehicles = data['vehicles']

# Keep one dictionary of all files for repeated inspection loops.
all_dfs = {
    'app_events': app_events,
    'complaints': complaints,
    'customers': customers,
    'data_dict': data_dict,
    'deliveries': deliveries,
    'drivers': drivers,
    'hubs': hubs,
    'incidents': incidents,
    'orders': orders,
    'vehicles': vehicles
}

print('All raw datasets are now available as individual dataframes.')


app_events: 640 rows, 10 columns loaded from GitHub
complaints: 320 rows, 10 columns loaded from GitHub
customers: 650 rows, 9 columns loaded from GitHub
data_dictionary: 9 rows, 3 columns loaded from GitHub
deliveries: 950 rows, 13 columns loaded from GitHub
drivers: 170 rows, 8 columns loaded from GitHub
hubs: 8 rows, 5 columns loaded from GitHub
incidents: 280 rows, 7 columns loaded from GitHub
orders: 1250 rows, 11 columns loaded from GitHub
vehicles: 120 rows, 8 columns loaded from GitHub
All raw datasets are now available as individual dataframes.


**Expected row counts:**  
customers = 650, orders = 1250, deliveries = 950, drivers = 170, vehicles = 120, hubs = 8, incidents = 280, complaints = 320, app_events = 640.

One early point to notice is that there are 1,250 orders but only 950 delivery records. Later checks show that 300 orders do not have a matching delivery record. This is useful for the report because it reflects the case study issue of fragmented and incomplete operational tracking.

---
## **Inspect Columns and Data Types**

The column names and data types for each file are reviewed. This helps identify columns that need cleaning or conversion, such as date and time fields imported as text.

In [ ]:
# Print column names and data types for every DataFrame
for name, df in all_dfs.items():
    print(f'\n=== {name.upper()} ===')
    print(df.dtypes.to_string())


=== APP_EVENTS ===
event_id           object
customer_id        object
order_id           object
event_timestamp    object
event_type         object
session_id         object
device_type        object
zone_context       object
api_latency_ms      int64
success_flag        int64

=== COMPLAINTS ===
complaint_id            object
customer_id             object
order_id                object
complaint_type          object
channel                 object
severity                object
created_at              object
status                  object
resolution_days          int64
compensation_amount    float64

=== CUSTOMERS ===
customer_id              object
age                       int64
home_zone                object
customer_type            object
signup_date              object
loyalty_score           float64
app_engagement_score    float64
preferred_channel        object
account_status           object

=== DATA_DICT ===
file_name       object
record_count     int64
description     ob

In [ ]:
# Preview the first 3 rows of each file to see real values
for name, df in all_dfs.items():
    if name == 'data_dict':
        continue  # Skip the data dictionary: it is metadata, not operational data
    print(f'\n=== {name.upper()}: First 3 rows ===')
    display(df.head(3))


=== APP_EVENTS: First 3 rows ===


,event_id,customer_id,order_id,event_timestamp,event_type,session_id,device_type,zone_context,api_latency_ms,success_flag
0,AE00001,C0488,NaN,2024-08-09 03:25:00,eta_refresh,S19847,Android,north,301,1
1,AE00002,C0595,O00950,2024-02-13 22:29:00,search_route,S32766,Android,SOUTH,60,1
2,AE00003,C0494,O00170,2025-08-11 09:29:00,chat_opened,S99516,iOS,Airport,1118,1



=== COMPLAINTS: First 3 rows ===


,complaint_id,customer_id,order_id,complaint_type,channel,severity,created_at,status,resolution_days,compensation_amount
0,CP0001,C0464,O00814,AppIssue,App,High,2025-03-30 02:36:00,Open,11,23.99
1,CP0002,C0056,O00628,MissedPickup,Phone,Medium,2024-11-07 10:05:00,Open,4,21.64
2,CP0003,C0469,O00384,Delay,Chatbot,High,2024-01-02 15:47:00,Open,16,26.41



=== CUSTOMERS: First 3 rows ===


...

,incident_id,delivery_id,incident_type,reported_at,severity,resolution_status,resolved_hours
0,I0001,DL00221,BatteryAlert,2024-03-11 23:46:00,Medium,Escalated,12.3
1,I0002,DL00578,BatteryAlert,2024-02-21 10:56:00,Low,Open,9.6
2,I0003,DL00175,TemperatureIssue,2025-04-17 23:22:00,Medium,Open,22.0



=== ORDERS: First 3 rows ===


,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
0,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
1,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
2,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0



=== VEHICLES: First 3 rows ===


,vehicle_id,vehicle_type,assigned_zone,commission_date,battery_health_pct,odometer_km,maintenance_status,telematics_version
0,V001,EV,North,2024-12-28 23:48:00,71.8,56928,Active,v2.2
1,V002,EV,AIRPORT,2024-04-21 16:14:00,67.9,159368,InRepair,v2.2
2,V003,CargoVan,north,2025-11-24 23:59:00,91.7,219359,Active,v2.1


---
## **Check Missing Values**

Missing values are checked before cleaning. This is important because missing data can affect statistics, charts, and joins. The aim here is not to fill everything automatically, but to understand where values are missing and decide how each case should be handled.

In [ ]:
# For each file, show count and percentage of missing values per column
for name, df in all_dfs.items():
    if name == 'data_dict':
        continue
    missing_count = df.isnull().sum()
    missing_pct   = (missing_count / len(df) * 100).round(1)
    summary = pd.DataFrame({
        'Missing Count': missing_count,
        'Missing %':     missing_pct
    })
    # Only show columns that actually have missing values
    summary = summary[summary['Missing Count'] > 0]
    if summary.empty:
        print(f'\n=== {name.upper()}: No missing values found ===')
    else:
        print(f'\n=== {name.upper()}: Missing values ===')
        print(summary.to_string())


=== APP_EVENTS: Missing values ===
          Missing Count  Missing %
order_id            144       22.5

=== COMPLAINTS: Missing values ===
                     Missing Count  Missing %
compensation_amount             16        5.0

=== CUSTOMERS: Missing values ===
                   Missing Count  Missing %
loyalty_score                 20        3.1
preferred_channel             13        2.0

=== DELIVERIES: Missing values ===
                               Missing Count  Missing %
delivery_completed_at                     19        2.0
customer_rating_post_delivery             14        1.5

=== DRIVERS: Missing values ===
                Missing Count  Missing %
training_score              7        4.1

=== HUBS: No missing values found ===

=== INCIDENTS: Missing values ===
                Missing Count  Missing %
resolved_hours             17        6.1

=== ORDERS: Missing values ===
                 Missing Count  Missing %
booking_channel             25        2.0

=== VEH

**Initial interpretation of missing values**

- `app_events.order_id` is missing in many rows. This is acceptable because not every app event, such as login or search activity, needs to be linked to a specific order.
- `customers.loyalty_score` and `preferred_channel` have missing values, so these need sensible handling before analysis.
- `orders.booking_channel` has missing values and can be filled as `Unknown`.
- `deliveries.delivery_completed_at` and `customer_rating_post_delivery` are missing in some records. These missing values may reflect incomplete service recording or cases where customers did not leave a rating.
- `drivers.training_score` and `vehicles.battery_health_pct` have missing numerical values. These can be filled using median values so the analysis is not distorted by extreme values.
- `complaints.compensation_amount` is missing in some rows. In this context, it is reasonable to treat missing compensation as 0.
- `incidents.resolved_hours` is missing for some records. This is left as missing because filling it would create artificial resolution times.

---
## **Check for Duplicate Rows**

Duplicate rows can make totals and averages inaccurate. This step checks whether any file contains fully duplicated records.

In [ ]:
for name, df in all_dfs.items():
    if name == 'data_dict':
        continue
    n_dupes = df.duplicated().sum()
    status = 'DUPLICATES FOUND: investigate' if n_dupes > 0 else 'No duplicates'
    print(f'{name:<15}: {n_dupes:>4} duplicate rows: {status}')

app_events     :    0 duplicate rows: No duplicates
complaints     :    0 duplicate rows: No duplicates
customers      :    0 duplicate rows: No duplicates
deliveries     :    0 duplicate rows: No duplicates
drivers        :    0 duplicate rows: No duplicates
hubs           :    0 duplicate rows: No duplicates
incidents      :    0 duplicate rows: No duplicates
orders         :    0 duplicate rows: No duplicates
vehicles       :    0 duplicate rows: No duplicates


---
## **Check Cross-File Relationships**

The NorthStar data is spread across multiple files, so it is important to check whether the key ID columns connect correctly. This step confirms whether the files can be safely joined for later analysis.

Examples of expected relationships include:

- `orders.customer_id` should match `customers.customer_id`
- `deliveries.order_id` should match `orders.order_id`
- `complaints.order_id` should match `orders.order_id`
- `incidents.delivery_id` should match `deliveries.delivery_id`

This directly supports the coursework aim of linking fragmented operational records together.

In [ ]:
# Relationship checks between files
# This confirms whether key IDs in one file exist in the related master file.

def relationship_check(left_df, left_col, right_df, right_col, relationship_name):
    left_values = set(left_df[left_col].dropna().astype(str))
    right_values = set(right_df[right_col].dropna().astype(str))
    missing_matches = left_values - right_values

    return {
        "relationship": relationship_name,
        "left_unique_values": len(left_values),
        "right_unique_values": len(right_values),
        "missing_matches": len(missing_matches),
        "status": "OK" if len(missing_matches) == 0 else "ISSUES FOUND"
    }

relationship_results = []

relationship_results.append(relationship_check(
    orders, "customer_id",
    customers, "customer_id",
    "orders.customer_id -> customers.customer_id"
))

relationship_results.append(relationship_check(
    deliveries, "order_id",
    orders, "order_id",
    "deliveries.order_id -> orders.order_id"
))

relationship_results.append(relationship_check(
    deliveries, "driver_id",
    drivers, "driver_id",
    "deliveries.driver_id -> drivers.driver_id"
))

relationship_results.append(relationship_check(
    deliveries, "vehicle_id",
    vehicles, "vehicle_id",
    "deliveries.vehicle_id -> vehicles.vehicle_id"
))

relationship_results.append(relationship_check(
    deliveries, "hub_id",
    hubs, "hub_id",
    "deliveries.hub_id -> hubs.hub_id"
))

relationship_results.append(relationship_check(
    complaints, "order_id",
    orders, "order_id",
    "complaints.order_id -> orders.order_id"
))

relationship_results.append(relationship_check(
    incidents, "delivery_id",
    deliveries, "delivery_id",
    "incidents.delivery_id -> deliveries.delivery_id"
))
relationship_df = pd.DataFrame(relationship_results)
relationship_df

,relationship,left_unique_values,right_unique_values,missing_matches,status
0,orders.customer_id -> customers.customer_id,568,650,0,OK
1,deliveries.order_id -> orders.order_id,950,1250,0,OK
2,deliveries.driver_id -> drivers.driver_id,170,170,0,OK
3,deliveries.vehicle_id -> vehicles.vehicle_id,120,120,0,OK
4,deliveries.hub_id -> hubs.hub_id,8,8,0,OK
5,complaints.order_id -> orders.order_id,285,1250,0,OK
6,incidents.delivery_id -> deliveries.delivery_id,248,950,0,OK


---
## **Inspect and Standardise Zone Names**

Zone names appear in several files, including customer, order, driver, vehicle, hub, and app event data. Since NorthStar is described as having fragmented systems, the same zone appears in different formats. For example, `Airport`, `AIRPORT`, and `airport` all refer to the same zone.

If these values are not standardised, zone-based analysis will be unreliable because one real zone may be split into several separate groups. This step checks the original values and then creates cleaned zone columns.

In [ ]:
# Collect all zone-type columns across all files and show their unique values
zone_columns = {
    'customers.home_zone':      customers['home_zone'],
    'orders.pickup_zone':       orders['pickup_zone'],
    'orders.dropoff_zone':      orders['dropoff_zone'],
    'drivers.base_zone':        drivers['base_zone'],
    'vehicles.assigned_zone':   vehicles['assigned_zone'],
    'hubs.zone':                hubs['zone'],
    'app_events.zone_context':  app_events['zone_context'],
}

for col_label, series in zone_columns.items():
    unique_vals = sorted(series.dropna().unique(), key=lambda x: x.lower())
    print(f'{col_label}: {unique_vals}')

customers.home_zone: ['AIRPORT', 'Airport', 'CENTRAL', 'Central', 'Ctr', 'East', 'EAST', 'North', 'north', 'NORTH', 'Riverside', 'RiverSide', 'South', 'SOUTH', 'WEST', 'West']
orders.pickup_zone: ['Airport', 'AIRPORT', 'CENTRAL', 'Central', 'Ctr', 'East', 'EAST', 'North', 'NORTH', 'north', 'RiverSide', 'Riverside', 'South', 'SOUTH', 'West', 'WEST']
orders.dropoff_zone: ['AIRPORT', 'Airport', 'Central', 'CENTRAL', 'Ctr', 'East', 'EAST', 'North', 'north', 'NORTH', 'Riverside', 'RiverSide', 'South', 'SOUTH', 'West', 'WEST']
drivers.base_zone: ['AIRPORT', 'Airport', 'Central', 'CENTRAL', 'Ctr', 'East', 'EAST', 'north', 'North', 'NORTH', 'Riverside', 'RiverSide', 'SOUTH', 'South', 'West', 'WEST']
vehicles.assigned_zone: ['AIRPORT', 'Airport', 'Central', 'CENTRAL', 'Ctr', 'EAST', 'East', 'North', 'north', 'NORTH', 'RiverSide', 'Riverside', 'South', 'SOUTH', 'West', 'WEST']
hubs.zone: ['Airport', 'Central', 'East', 'North', 'Riverside', 'South', 'West']
app_events.zone_context: ['Airport', 'A

In [ ]:
# Define the standardisation mapping.
# Keys are all known dirty variants. Values are the clean canonical form.
ZONE_MAP = {
    # Airport variants
    'AIRPORT':   'Airport',
    'airport':   'Airport',
    'Airport':   'Airport',

    # North variants
    'NORTH':     'North',
    'north':     'North',
    'North':     'North',

    # South variants
    'SOUTH':     'South',
    'south':     'South',
    'South':     'South',

    # Central variants
    'CENTRAL':   'Central',
    'central':   'Central',
    'Central':   'Central',
    'Ctr':       'Central',
    'ctr':       'Central',
    'CTR':       'Central',

    # East variants
    'EAST':      'East',
    'east':      'East',
    'East':      'East',

    # West variants
    'WEST':      'West',
    'west':      'West',
    'West':      'West',

    # Riverside variants
    'RIVERSIDE': 'Riverside',
    'riverside': 'Riverside',
    'Riverside': 'Riverside',
    'RiverSide': 'Riverside',
    'Riveride':  'Riverside',   # typo variant
}

# Helper function: apply the map to a Series, leave unrecognised values as-is
def clean_zone(series):
    """Map dirty zone values to canonical form. Unknown values are kept unchanged."""
    return series.map(lambda x: ZONE_MAP.get(x, x) if pd.notnull(x) else x)

print('Zone standardisation map defined. Unique canonical zones will be:')
print(sorted(set(ZONE_MAP.values())))

Zone standardisation map defined. Unique canonical zones will be:
['Airport', 'Central', 'East', 'North', 'Riverside', 'South', 'West']


In [ ]:
# Apply zone cleaning to each file: adding a new '_clean' column alongside the original
# so we can always check what changed.

customers['home_zone_clean']       = clean_zone(customers['home_zone'])
orders['pickup_zone_clean']        = clean_zone(orders['pickup_zone'])
orders['dropoff_zone_clean']       = clean_zone(orders['dropoff_zone'])
drivers['base_zone_clean']         = clean_zone(drivers['base_zone'])
vehicles['assigned_zone_clean']    = clean_zone(vehicles['assigned_zone'])
hubs['zone_clean']                 = clean_zone(hubs['zone'])
app_events['zone_context_clean']   = clean_zone(app_events['zone_context'])

# Verify: show unique values after cleaning for the most important columns
print('customers.home_zone_clean:    ', sorted(customers['home_zone_clean'].dropna().unique()))
print('orders.pickup_zone_clean:     ', sorted(orders['pickup_zone_clean'].dropna().unique()))
print('orders.dropoff_zone_clean:    ', sorted(orders['dropoff_zone_clean'].dropna().unique()))

customers.home_zone_clean:     ['Airport', 'Central', 'East', 'North', 'Riverside', 'South', 'West']
orders.pickup_zone_clean:      ['Airport', 'Central', 'East', 'North', 'Riverside', 'South', 'West']
orders.dropoff_zone_clean:     ['Airport', 'Central', 'East', 'North', 'Riverside', 'South', 'West']


---
##  **Convert Date and Time Columns**

Date and time columns are imported as text by default. They need to be converted into Pandas datetime format so that durations, time patterns, and ordering can be analysed correctly.

In [ ]:
# customers
customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

# orders
orders['order_created_at'] = pd.to_datetime(orders['order_created_at'], errors='coerce')

# deliveries
deliveries['dispatch_time']          = pd.to_datetime(deliveries['dispatch_time'], errors='coerce')
deliveries['delivery_completed_at']  = pd.to_datetime(deliveries['delivery_completed_at'], errors='coerce')

# vehicles
vehicles['commission_date'] = pd.to_datetime(vehicles['commission_date'], errors='coerce')

# incidents
incidents['reported_at'] = pd.to_datetime(incidents['reported_at'], errors='coerce')

# complaints
complaints['created_at'] = pd.to_datetime(complaints['created_at'], errors='coerce')

# app_events
app_events['event_timestamp'] = pd.to_datetime(app_events['event_timestamp'], errors='coerce')

# Confirm conversion worked: should show dtype datetime64[ns]
print('orders.order_created_at dtype:         ', orders['order_created_at'].dtype)
print('deliveries.dispatch_time dtype:        ', deliveries['dispatch_time'].dtype)
print('deliveries.delivery_completed_at dtype:', deliveries['delivery_completed_at'].dtype)
print('complaints.created_at dtype:           ', complaints['created_at'].dtype)
print('app_events.event_timestamp dtype:      ', app_events['event_timestamp'].dtype)

orders.order_created_at dtype:          datetime64[ns]
deliveries.dispatch_time dtype:         datetime64[ns]
deliveries.delivery_completed_at dtype: datetime64[ns]
complaints.created_at dtype:            datetime64[ns]
app_events.event_timestamp dtype:       datetime64[ns]


---
## **Handle Missing Values**

Missing values are handled based on the meaning of each column. A single rule is not applied to every missing value because different fields require different treatment. For example, a missing booking channel can be labelled as `Unknown`, but a missing completion time should not be invented.

In [ ]:
# --- customers ---
# loyalty_score: fill with median (typical score for a customer with no recorded score)
loyalty_median = customers['loyalty_score'].median()
customers['loyalty_score'] = customers['loyalty_score'].fillna(loyalty_median)

# preferred_channel: fill with 'Unknown' (we do not know their preference)
customers['preferred_channel'] = customers['preferred_channel'].fillna('Unknown')

print('customers: missing after fill:')
print(customers[['loyalty_score', 'preferred_channel']].isnull().sum())

customers: missing after fill:
loyalty_score        0
preferred_channel    0
dtype: int64


In [ ]:
# --- orders ---
# booking_channel: fill with 'Unknown'
orders['booking_channel'] = orders['booking_channel'].fillna('Unknown')

print('orders: missing after fill:')
print(orders[['booking_channel']].isnull().sum())

orders: missing after fill:
booking_channel    0
dtype: int64


In [ ]:
# --- deliveries ---
# delivery_completed_at: leave as NaN: a missing completion time for a Failed/Delayed
# delivery is meaningful (it means the service was not completed normally).
# We will capture this in an engineered feature below.

# customer_rating_post_delivery: leave as NaN: an unrated delivery is different from
# a rated one; we should not invent a score. We will handle this by filtering in analyses.

print('deliveries: missing values (informational):')
print(deliveries[['delivery_completed_at', 'customer_rating_post_delivery']].isnull().sum())

deliveries: missing values (informational):
delivery_completed_at            19
customer_rating_post_delivery    14
dtype: int64


In [ ]:
# --- drivers ---
# training_score: fill with median (best neutral estimate for unknown scores)
training_median = drivers['training_score'].median()
drivers['training_score'] = drivers['training_score'].fillna(training_median)

print(f'drivers: training_score median used for fill: {training_median}')
print('drivers: missing after fill:')
print(drivers['training_score'].isnull().sum())

drivers: training_score median used for fill: 75.2
drivers: missing after fill:
0


In [ ]:
# --- vehicles ---
# battery_health_pct: fill with median (reasonable fleet-level assumption)
battery_median = vehicles['battery_health_pct'].median()
vehicles['battery_health_pct'] = vehicles['battery_health_pct'].fillna(battery_median)

print(f'vehicles: battery_health_pct median used for fill: {battery_median}')
print('vehicles: missing after fill:')
print(vehicles['battery_health_pct'].isnull().sum())

vehicles: battery_health_pct median used for fill: 78.05
vehicles: missing after fill:
0


In [ ]:
# --- complaints ---
# compensation_amount: fill with 0.0: missing most likely means no compensation was given
complaints['compensation_amount'] = complaints['compensation_amount'].fillna(0.0)

print('complaints: missing after fill:')
print(complaints['compensation_amount'].isnull().sum())

complaints: missing after fill:
0


In [ ]:
# --- incidents ---
# resolved_hours: leave as NaN: it genuinely means the incident is not yet resolved.
# Filling it would create false data about resolution time.
print('incidents: resolved_hours missing (still-open incidents):')
print(incidents['resolved_hours'].isnull().sum())
print('Total incidents:', len(incidents))

incidents: resolved_hours missing (still-open incidents):
17
Total incidents: 280


In [ ]:
# --- app_events ---
# order_id: leave as NaN: many app events (e.g. search_route, chat_escalated)
# are not linked to a specific order. This is a valid operational reality.
print('app_events: order_id missing (events without a linked order):')
print(app_events['order_id'].isnull().sum())

app_events: order_id missing (events without a linked order):
144


---
## **Engineer Useful Features**

Feature engineering creates new columns from the raw data. These new features are useful because they translate the dataset into variables that match the business problems in the case study, such as delay, failure, cost, missed promised windows, complaints, incidents, and data quality issues.

In [ ]:
# ---- DELIVERY FEATURES ----

# is_delayed: 1 if delivery was Delayed, 0 otherwise
deliveries['is_delayed'] = (deliveries['delivery_status'] == 'Delayed').astype(int)

# is_failed: 1 if delivery Failed completely
deliveries['is_failed'] = (deliveries['delivery_status'] == 'Failed').astype(int)

# is_bad_outcome: 1 if Delayed OR Failed: this captures all non-successful deliveries
deliveries['is_bad_outcome'] = (
    deliveries['delivery_status'].isin(['Delayed', 'Failed'])
).astype(int)

# completion_hours: how long did the delivery take from dispatch to completion?
# Will be NaN for Failed/unfinished deliveries: that is expected.
deliveries['completion_hours'] = (
    deliveries['delivery_completed_at'] - deliveries['dispatch_time']
).dt.total_seconds() / 3600

# negative_completion_time_flag: 1 if completion time is earlier than dispatch time
# This is not realistic and is treated as a data quality issue.
deliveries['negative_completion_time_flag'] = (
    deliveries['completion_hours'] < 0
).astype(int)

# completion_hours_clean removes impossible negative durations from analysis
# We keep the original completion_hours column for audit purposes.
deliveries['completion_hours_clean'] = deliveries['completion_hours'].where(
    deliveries['completion_hours'] >= 0,
    np.nan
)

print("Negative completion time records:")
print(deliveries['negative_completion_time_flag'].value_counts())

# cost_per_km: fuel/charge cost divided by route distance: a cost efficiency metric
# Avoid division by zero with np.where
deliveries['cost_per_km'] = np.where(
    deliveries['route_distance_km'] > 0,
    deliveries['fuel_or_charge_cost'] / deliveries['route_distance_km'],
    np.nan
)

print('Delivery features added:')
print(deliveries[['delivery_id', 'delivery_status', 'is_delayed', 'is_failed',
                  'is_bad_outcome', 'completion_hours', 'negative_completion_time_flag',
                  'completion_hours_clean', 'cost_per_km']].head(5))

Negative completion time records:
negative_completion_time_flag
0    886
1     64
Name: count, dtype: int64
Delivery features added:
  delivery_id delivery_status  is_delayed  is_failed  is_bad_outcome  completion_hours  negative_completion_time_flag  completion_hours_clean  cost_per_km
0     DL00001          Failed           0          1               1         22.149973                              0               22.149973     0.698146
1     DL00002          OnTime           0          0               0         -1.100000                              1                     NaN     1.296905
2     DL00003          OnTime           0          0               0          1.108991                              0                1.108991     1.074495
3     DL00004         Delayed           1          0               1         23.985584                              0               23.985584     0.829476
4     DL00005          OnTime           0          0               0          4.042814      

In [ ]:
# ---- ORDER FEATURES ----

# order_month: extract the month from order_created_at for time-series analysis
orders['order_month'] = orders['order_created_at'].dt.to_period('M').astype(str)

# order_hour: extract hour of day: useful for identifying peak service times
orders['order_hour'] = orders['order_created_at'].dt.hour

print('Order time features added:')
print(orders[['order_id', 'order_created_at', 'order_month', 'order_hour']].head(5))

Order time features added:
  order_id    order_created_at order_month  order_hour
0   O00001 2024-08-20 14:43:00     2024-08          14
1   O00002 2024-05-14 22:16:00     2024-05          22
2   O00003 2025-09-02 14:37:00     2025-09          14
3   O00004 2025-01-11 17:15:00     2025-01          17
4   O00005 2025-02-17 19:32:00     2025-02          19


In [ ]:
# ---- APP EVENT FEATURES ----

# api_latency_category: classify latency into bands for easier analysis
def categorise_latency(ms):
    if pd.isnull(ms):
        return 'Unknown'
    elif ms < 100:
        return 'Fast (<100ms)'
    elif ms < 300:
        return 'Moderate (100-299ms)'
    else:
        return 'Slow (>=300ms)'

app_events['api_latency_category'] = app_events['api_latency_ms'].apply(categorise_latency)

print('App event latency categories:')
print(app_events['api_latency_category'].value_counts())

App event latency categories:
api_latency_category
Slow (>=300ms)          458
Moderate (100-299ms)    137
Fast (<100ms)            45
Name: count, dtype: int64


In [ ]:
# ---- COMPLAINT FEATURES ----

# complaint_resolved_flag: 1 if status is Resolved, 0 otherwise
complaints['complaint_resolved_flag'] = (
    complaints['status'] == 'Resolved'
).astype(int)

print('Complaint resolution distribution:')
print(complaints['status'].value_counts())
print('complaint_resolved_flag distribution:')
print(complaints['complaint_resolved_flag'].value_counts())

Complaint resolution distribution:
status
Resolved            186
Open                 56
AwaitingCustomer     40
Escalated            38
Name: count, dtype: int64
complaint_resolved_flag distribution:
complaint_resolved_flag
1    186
0    134
Name: count, dtype: int64


In [ ]:
# ---- VEHICLE FEATURES ----

# low_battery_flag: 1 if battery health is below 60%: a maintenance risk threshold
vehicles['low_battery_flag'] = (vehicles['battery_health_pct'] < 60).astype(int)

# in_repair_flag: 1 if vehicle is currently in maintenance/repair
vehicles['in_repair_flag'] = (vehicles['maintenance_status'] == 'InRepair').astype(int)

print('Vehicle flags:')
print(vehicles[['vehicle_id', 'battery_health_pct', 'low_battery_flag',
                'maintenance_status', 'in_repair_flag']].head(8))

Vehicle flags:
  vehicle_id  battery_health_pct  low_battery_flag maintenance_status  in_repair_flag
0       V001               71.80                 0             Active               0
1       V002               67.90                 0           InRepair               1
2       V003               91.70                 0             Active               0
3       V004               78.05                 0             Active               0
4       V005               58.60                 1             Active               0
5       V006               78.60                 0             Active               0
6       V007               68.60                 0             Active               0
7       V008               90.50                 0             Active               0


---
## **Create Complaint and Incident Flags**

The case study says that NorthStar struggles to connect customer complaints, failed services, journeys, and incidents into one view. To support that analysis, two simple indicators are created:

- `has_complaint`: shows whether an order has at least one complaint;
- `has_incident`: shows whether a delivery has at least one incident.

These flags make it easier to connect customer experience problems with operational records.

In [ ]:
# Create flags showing whether an order has a complaint and whether a delivery has an incident

complaint_orders = set(complaints["order_id"].dropna())
incident_deliveries = set(incidents["delivery_id"].dropna())

orders["has_complaint"] = orders["order_id"].isin(complaint_orders).astype(int)
deliveries["has_incident"] = deliveries["delivery_id"].isin(incident_deliveries).astype(int)

print("Orders with complaint flag:")
print(orders["has_complaint"].value_counts())

print("\nDeliveries with incident flag:")
print(deliveries["has_incident"].value_counts())

Orders with complaint flag:
has_complaint
0    965
1    285
Name: count, dtype: int64

Deliveries with incident flag:
has_incident
0    702
1    248
Name: count, dtype: int64


---
## **Build Merged Operational Datasets**

The separate CSV files each show only one part of NorthStar’s operations. To understand the bigger picture, merged datasets are created by combining related files.

The main merged datasets are:

1. **`ops_df`**: the main operational dataset combining orders, deliveries, customers, hubs, drivers, and vehicles;
2. **`complaint_df`**: a complaint-focused dataset combining complaints with related order and delivery details;
3. **`incident_df`**: an incident-focused dataset combining incidents with delivery, vehicle, and hub information.

These merged files will be used in later analysis sections.

In [ ]:
# ---- MERGE 1: Main operational DataFrame ----
# Start with orders, then left-join deliveries (not all orders have a delivery)
ops_df = orders.merge(deliveries,  on='order_id',  how='left')

# Join customer data
ops_df = ops_df.merge(
    customers[['customer_id', 'age', 'home_zone_clean', 'customer_type',
               'loyalty_score', 'app_engagement_score', 'preferred_channel', 'account_status']],
    on='customer_id', how='left'
)

# Join hub data
ops_df = ops_df.merge(
    hubs[['hub_id', 'hub_name', 'zone_clean', 'hub_type', 'capacity_score']],
    on='hub_id', how='left'
)

# Join driver data
ops_df = ops_df.merge(
    drivers[['driver_id', 'base_zone_clean', 'employment_type', 'years_experience',
             'training_score', 'driver_rating']],
    on='driver_id', how='left'
)

# Join vehicle data
ops_df = ops_df.merge(
    vehicles[['vehicle_id', 'vehicle_type', 'assigned_zone_clean', 'battery_health_pct',
              'maintenance_status', 'low_battery_flag', 'in_repair_flag']],
    on='vehicle_id', how='left'
)

print(f'Main operational DataFrame shape: {ops_df.shape}')
print(f'Columns: {list(ops_df.columns)}')

# missed_promised_window checks whether the delivery took longer than the promised service window

ops_df["missed_promised_window"] = (
    ops_df["completion_hours_clean"] > ops_df["promised_window_hours"]
).astype(float)

# If completion_hours_clean is missing, keep missed_promised_window as missing instead of forcing it to 0
ops_df.loc[ops_df["completion_hours_clean"].isna(), "missed_promised_window"] = np.nan

ops_df[[
    "order_id",
    "service_type",
    "promised_window_hours",
    "completion_hours",
    "completion_hours_clean",
    "missed_promised_window",
    "delivery_status"
]].head()

Main operational DataFrame shape: (1250, 58)
Columns: ['order_id', 'customer_id', 'service_type', 'order_created_at', 'promised_window_hours', 'pickup_zone', 'dropoff_zone', 'priority_level', 'order_value', 'booking_channel', 'special_handling_flag', 'pickup_zone_clean', 'dropoff_zone_clean', 'order_month', 'order_hour', 'has_complaint', 'delivery_id', 'driver_id', 'vehicle_id', 'hub_id', 'dispatch_time', 'delivery_completed_at', 'delivery_status', 'route_distance_km', 'manual_route_override_count', 'proof_of_completion_missing', 'customer_rating_post_delivery', 'fuel_or_charge_cost', 'is_delayed', 'is_failed', 'is_bad_outcome', 'completion_hours', 'negative_completion_time_flag', 'completion_hours_clean', 'cost_per_km', 'has_incident', 'age', 'home_zone_clean', 'customer_type', 'loyalty_score', 'app_engagement_score', 'preferred_channel', 'account_status', 'hub_name', 'zone_clean', 'hub_type', 'capacity_score', 'base_zone_clean', 'employment_type', 'years_experience', 'training_score'

,order_id,service_type,promised_window_hours,completion_hours,completion_hours_clean,missed_promised_window,delivery_status
0,O00001,Passenger,6,2.398937,2.398937,0.0,OnTime
1,O00002,Passenger,24,NaN,NaN,NaN,NaN
2,O00003,Passenger,4,8.861012,8.861012,1.0,Delayed
3,O00004,Parcel,2,-1.100000,NaN,NaN,OnTime
4,O00005,Retail,12,11.700013,11.700013,0.0,OnTime


In [ ]:
# ---- MERGE 2: Complaint-focused DataFrame ----
complaint_df = complaints.merge(
    orders[['order_id', 'service_type', 'pickup_zone_clean', 'dropoff_zone_clean',
            'order_value', 'order_created_at']],
    on='order_id', how='left'
)
complaint_df = complaint_df.merge(
    deliveries[['order_id', 'delivery_status', 'hub_id', 'driver_id',
                'is_bad_outcome', 'manual_route_override_count', 'customer_rating_post_delivery']],
    on='order_id', how='left'
)

print(f'Complaint DataFrame shape: {complaint_df.shape}')
print(f'Columns: {list(complaint_df.columns)}')

Complaint DataFrame shape: (320, 22)
Columns: ['complaint_id', 'customer_id', 'order_id', 'complaint_type', 'channel', 'severity', 'created_at', 'status', 'resolution_days', 'compensation_amount', 'complaint_resolved_flag', 'service_type', 'pickup_zone_clean', 'dropoff_zone_clean', 'order_value', 'order_created_at', 'delivery_status', 'hub_id', 'driver_id', 'is_bad_outcome', 'manual_route_override_count', 'customer_rating_post_delivery']


In [ ]:
# ---- MERGE 3: Incident-focused DataFrame ----
incident_df = incidents.merge(
    deliveries[['delivery_id', 'order_id', 'driver_id', 'vehicle_id', 'hub_id',
                'delivery_status', 'is_bad_outcome', 'route_distance_km']],
    on='delivery_id', how='left'
)
incident_df = incident_df.merge(
    vehicles[['vehicle_id', 'vehicle_type', 'battery_health_pct', 'maintenance_status',
              'low_battery_flag']],
    on='vehicle_id', how='left'
)
incident_df = incident_df.merge(
    hubs[['hub_id', 'hub_name', 'zone_clean']],
    on='hub_id', how='left'
)

print(f'Incident DataFrame shape: {incident_df.shape}')
print(f'Columns: {list(incident_df.columns)}')

Incident DataFrame shape: (280, 20)
Columns: ['incident_id', 'delivery_id', 'incident_type', 'reported_at', 'severity', 'resolution_status', 'resolved_hours', 'order_id', 'driver_id', 'vehicle_id', 'hub_id', 'delivery_status', 'is_bad_outcome', 'route_distance_km', 'vehicle_type', 'battery_health_pct', 'maintenance_status', 'low_battery_flag', 'hub_name', 'zone_clean']


---
## **Verify Data Integrity After Merging**

After merging datasets, the data integrity check confirms the row counts to make sure the process did not accidentally remove or duplicate records. This is a useful check because wrong joins can produce misleading analysis.

In [ ]:
print('=== Data Integrity Check After Merging ===')
print()
print(f'orders rows:                   {len(orders)}')
print(f'ops_df rows (orders-based):    {len(ops_df)}')
print(f'  -> Expected: 1250 (left join keeps all orders, delivery columns NaN for unmatched)')
print()
print(f'complaints rows:               {len(complaints)}')
print(f'complaint_df rows:             {len(complaint_df)}')
print(f'  -> Should be equal to complaints rows (left join from complaints)')
print()
print(f'incidents rows:                {len(incidents)}')
print(f'incident_df rows:              {len(incident_df)}')
print(f'  -> Should be equal to incidents rows (left join from incidents)')
print()

# Check: how many orders have NO delivery record?
orders_without_delivery = ops_df['delivery_id'].isnull().sum()
print(f'Orders with no delivery record: {orders_without_delivery} out of {len(orders)}')
print('This confirms the case study observation that 300 orders are disconnected from delivery tracking.')

=== Data Integrity Check After Merging ===

orders rows:                   1250
ops_df rows (orders-based):    1250
  -> Expected: 1250 (left join keeps all orders, delivery columns NaN for unmatched)

complaints rows:               320
complaint_df rows:             320
  -> Should be equal to complaints rows (left join from complaints)

incidents rows:                280
incident_df rows:              280
  -> Should be equal to incidents rows (left join from incidents)

Orders with no delivery record: 300 out of 1250
This confirms the case study observation that 300 orders are disconnected from delivery tracking.


---
##  **Create Zone and Hub Performance Summary Tables**

The case study highlights that some zones and hubs may be performing worse than others. These summary tables help identify where delays, failures, complaints, higher costs, lower ratings, and manual route overrides are concentrated.

The outputs from this step can later be used in the Data Overview, Python Analytics, and Discussion sections of the report.

In [ ]:
# Zone-level performance summary
# This helps identify which pickup zones have worse operational outcomes.

zone_performance = ops_df.groupby("pickup_zone_clean").agg(
    total_orders=("order_id", "count"),
    delivery_records=("delivery_id", lambda x: x.notna().sum()),
    bad_outcomes=("is_bad_outcome", "sum"),
    complaints=("has_complaint", "sum"),
    avg_completion_hours=("completion_hours_clean", "mean"),
    avg_cost_per_km=("cost_per_km", "mean"),
    avg_rating=("customer_rating_post_delivery", "mean"),
    avg_manual_overrides=("manual_route_override_count", "mean")
).reset_index()

zone_performance["bad_outcome_rate"] = np.where(
    zone_performance["delivery_records"] > 0,
    zone_performance["bad_outcomes"] / zone_performance["delivery_records"],
    np.nan
)

zone_performance["complaint_rate"] = (
    zone_performance["complaints"] / zone_performance["total_orders"]
)

zone_performance = zone_performance.sort_values("bad_outcome_rate", ascending=False)

zone_performance

,pickup_zone_clean,total_orders,delivery_records,bad_outcomes,complaints,avg_completion_hours,avg_cost_per_km,avg_rating,avg_manual_overrides,bad_outcome_rate,complaint_rate
1,Central,238,174,84.0,56,11.245158,1.323963,3.546036,1.293103,0.482759,0.235294
0,Airport,144,113,43.0,28,9.648656,0.616088,3.984037,1.805310,0.380531,0.194444
4,Riverside,151,119,43.0,39,10.756360,1.417900,3.864492,0.731092,0.361345,0.258278
2,East,207,156,50.0,46,10.428183,1.359272,3.912078,0.788462,0.320513,0.222222
3,North,174,135,43.0,50,9.785670,1.338075,3.896667,0.696296,0.318519,0.287356
6,West,155,114,35.0,26,10.160827,1.248104,3.896316,0.807018,0.307018,0.167742
5,South,181,139,36.0,40,9.848314,1.402091,4.051825,0.690647,0.258993,0.220994


In [ ]:
# Hub-level performance summary
# This helps identify whether some operational hubs are linked to more failed or delayed services.

hub_performance = ops_df.groupby(["hub_id", "hub_name", "zone_clean"]).agg(
    delivery_records=("delivery_id", lambda x: x.notna().sum()),
    bad_outcomes=("is_bad_outcome", "sum"),
    avg_completion_hours=("completion_hours_clean", "mean"),
    avg_cost_per_km=("cost_per_km", "mean"),
    avg_rating=("customer_rating_post_delivery", "mean"),
    avg_manual_overrides=("manual_route_override_count", "mean")
).reset_index()

hub_performance["bad_outcome_rate"] = np.where(
    hub_performance["delivery_records"] > 0,
    hub_performance["bad_outcomes"] / hub_performance["delivery_records"],
    np.nan
)

hub_performance = hub_performance.sort_values("bad_outcome_rate", ascending=False)

hub_performance

,hub_id,hub_name,zone_clean,delivery_records,bad_outcomes,avg_completion_hours,avg_cost_per_km,avg_rating,avg_manual_overrides,bad_outcome_rate
4,H05,Central Core,Central,115,48.0,11.553300,1.477564,3.669558,0.947826,0.417391
5,H06,Airport Hub,Airport,104,42.0,9.917423,1.239235,3.882136,0.913462,0.403846
7,H08,Midtown Relay,Central,128,48.0,10.560490,1.181762,3.884560,1.109375,0.375000
3,H04,West Gate,West,127,44.0,11.102635,1.355406,3.915476,0.874016,0.346457
1,H02,South Link,South,106,36.0,9.478948,1.168068,3.950952,0.915094,0.339623
6,H07,Riverside Hub,Riverside,115,39.0,10.535888,1.222085,3.881858,1.052174,0.339130
0,H01,North Exchange,North,136,43.0,10.684968,1.371519,3.840593,1.029412,0.316176
2,H03,East Dock,East,119,34.0,8.437893,1.054579,3.895862,0.890756,0.285714


---
## **Descriptive Statistics Summary**

Before saving the cleaned data, descriptive statistics are generated for important numerical columns. This gives a quick check of the main ranges, averages, and possible unusual values in the dataset.

In [ ]:
print('=== Deliveries: Numerical Summary ===')
display(deliveries[['route_distance_km', 'manual_route_override_count',
                     'customer_rating_post_delivery', 'fuel_or_charge_cost',
                     'completion_hours', 'completion_hours_clean',
                     'negative_completion_time_flag', 'cost_per_km']].describe().round(2))

=== Deliveries: Numerical Summary ===


,route_distance_km,manual_route_override_count,customer_rating_post_delivery,fuel_or_charge_cost,completion_hours,completion_hours_clean,negative_completion_time_flag,cost_per_km
count,950.00,950.00,936.00,950.00,931.00,867.00,950.00,950.00
mean,13.91,0.97,3.86,12.84,9.55,10.32,0.07,1.26
std,7.48,1.09,0.89,4.34,8.65,8.46,0.25,1.24
min,1.20,0.00,1.00,2.50,-2.21,0.02,0.00,0.17
25%,9.14,0.00,3.36,9.93,2.95,3.50,0.00,0.70
50%,12.84,1.00,4.04,12.64,7.07,7.91,0.00,0.95
75%,16.84,2.00,4.55,15.70,14.64,15.53,0.00,1.33
max,41.94,7.00,5.00,29.43,43.46,43.46,1.00,12.36


In [ ]:
print('=== Orders: Service Type Distribution ===')
print(orders['service_type'].value_counts())
print()
print('=== Deliveries: Status Distribution ===')
print(deliveries['delivery_status'].value_counts())
print()
print('=== Complaints: Complaint Type Distribution ===')
print(complaints['complaint_type'].value_counts())

=== Orders: Service Type Distribution ===
service_type
Passenger    341
Parcel       308
Retail       297
Business     165
Medical      139
Name: count, dtype: int64

=== Deliveries: Status Distribution ===
delivery_status
OnTime     616
Delayed    202
Failed     132
Name: count, dtype: int64

=== Complaints: Complaint Type Distribution ===
complaint_type
Delay                101
MissedPickup          64
AppIssue              53
DriverBehaviour       51
SupportExperience     20
Billing               16
Damage                15
Name: count, dtype: int64


In [ ]:
print('=== Zone Distribution in Orders (Cleaned Pickup Zone) ===')
print(orders['pickup_zone_clean'].value_counts())
print()
print('=== Incident Types ===')
print(incidents['incident_type'].value_counts())

=== Zone Distribution in Orders (Cleaned Pickup Zone) ===
pickup_zone_clean
Central      238
East         207
South        181
North        174
West         155
Riverside    151
Airport      144
Name: count, dtype: int64

=== Incident Types ===
incident_type
ProofMissing        46
CustomerNoShow      44
RouteDeviation      43
VehicleFault        37
BatteryAlert        36
AppSyncError        31
TemperatureIssue    29
SafetyNearMiss      14
Name: count, dtype: int64


---
## **Check Missing Values After Cleaning**

After cleaning, some missing values are expected to remain. This is not always a problem. For example, a missing delivery completion time may show incomplete operational recording, while an app event without an order ID may simply be a general app interaction.

This step records what is still missing so that the remaining data limitations can be explained honestly in the report.

In [ ]:
# Missing values after cleaning
# Some missing values are intentionally kept because they carry business meaning.

cleaned_objects = {
    "customers": customers,
    "orders": orders,
    "deliveries": deliveries,
    "drivers": drivers,
    "vehicles": vehicles,
    "hubs": hubs,
    "incidents": incidents,
    "complaints": complaints,
    "app_events": app_events,
    "ops_df": ops_df,
    "complaint_df": complaint_df,
    "incident_df": incident_df,
    "zone_performance": zone_performance,
    "hub_performance": hub_performance
}

missing_after = []

for name, df in cleaned_objects.items():
    for col in df.columns:
        missing_count = df[col].isna().sum()
        if missing_count > 0:
            missing_after.append({
                "dataset": name,
                "column": col,
                "missing_count": missing_count,
                "missing_percent": round((missing_count / len(df)) * 100, 2)
            })

missing_after_df = pd.DataFrame(missing_after).sort_values(
    ["dataset", "missing_count"],
    ascending=[True, False]
)

missing_after_df

,dataset,column,missing_count,missing_percent
5,app_events,order_id,144,22.50
47,complaint_df,customer_rating_post_delivery,91,28.44
42,complaint_df,delivery_status,88,27.50
43,complaint_df,hub_id,88,27.50
44,complaint_df,driver_id,88,27.50
45,complaint_df,is_bad_outcome,88,27.50
46,complaint_df,manual_route_override_count,88,27.50
3,deliveries,completion_hours_clean,83,8.74
0,deliveries,delivery_completed_at,19,2.00
2,deliveries,completion_hours,19,2.00


---
## **Save Cleaned Files Locally for Coursework Evidence**

All cleaned and merged datasets are saved as CSV files inside the Colab session. This avoids repeating the same cleaning work in later notebooks and gives a consistent set of files for SQL in R, R analytics, Python analysis, MongoDB modelling, and query optimisation.

After this cell runs, the output files are zipped as cleaned coursework evidence for the `Data/Cleaned` folder.

In [ ]:
from pathlib import Path
import shutil

CLEANED_PATH = Path('/content/northstar_cleaned')
CLEANED_PATH.mkdir(parents=True, exist_ok=True)

# Save individual cleaned files
customers.to_csv(CLEANED_PATH / 'customers_clean.csv', index=False)
orders.to_csv(CLEANED_PATH / 'orders_clean.csv', index=False)
deliveries.to_csv(CLEANED_PATH / 'deliveries_clean.csv', index=False)
drivers.to_csv(CLEANED_PATH / 'drivers_clean.csv', index=False)
vehicles.to_csv(CLEANED_PATH / 'vehicles_clean.csv', index=False)
hubs.to_csv(CLEANED_PATH / 'hubs_clean.csv', index=False)
incidents.to_csv(CLEANED_PATH / 'incidents_clean.csv', index=False)
complaints.to_csv(CLEANED_PATH / 'complaints_clean.csv', index=False)
app_events.to_csv(CLEANED_PATH / 'app_events_clean.csv', index=False)

# Save merged dataframes
ops_df.to_csv(CLEANED_PATH / 'ops_merged.csv', index=False)
complaint_df.to_csv(CLEANED_PATH / 'complaints_merged.csv', index=False)
incident_df.to_csv(CLEANED_PATH / 'incidents_merged.csv', index=False)

# Save performance summary tables
zone_performance.to_csv(CLEANED_PATH / 'zone_performance_summary.csv', index=False)
hub_performance.to_csv(CLEANED_PATH / 'hub_performance_summary.csv', index=False)

print('All cleaned files saved locally in Colab:')
print(CLEANED_PATH)
print()

for file_path in sorted(CLEANED_PATH.glob('*.csv')):
    size_kb = file_path.stat().st_size // 1024
    print(f'{file_path.name:<35} {size_kb} KB')

# Create a zip file to archive the cleaned outputs
zip_file_base = '/content/northstar_cleaned_outputs'
zip_file = zip_file_base + '.zip'

if Path(zip_file).exists():
    Path(zip_file).unlink()

zip_file = shutil.make_archive(zip_file_base, 'zip', CLEANED_PATH)

print('\nCleaned output zip created:')
print(zip_file)

All cleaned files saved locally in Colab:
/content/northstar_cleaned

app_events_clean.csv                64 KB
complaints_clean.csv                25 KB
complaints_merged.csv               47 KB
customers_clean.csv                 46 KB
deliveries_clean.csv                154 KB
drivers_clean.csv                   8 KB
hub_performance_summary.csv         1 KB
hubs_clean.csv                      0 KB
incidents_clean.csv                 18 KB
incidents_merged.csv                39 KB
ops_merged.csv                      438 KB
orders_clean.csv                    124 KB
vehicles_clean.csv                  8 KB
zone_performance_summary.csv        1 KB

Cleaned output zip created:
/content/northstar_cleaned_outputs.zip


## **Summary of Data Audit and Cleaning**

The raw NorthStar files were imported from GitHub, reviewed for structure and quality, cleaned using Python and Pandas, and combined into operational datasets. The cleaned outputs include individual cleaned tables, merged operational data, complaint and incident relationship tables, and zone and hub performance summaries. These outputs provide the prepared dataset foundation for the later analysis and database implementation sections.